# GaiaLab governed LoRA training on Kaggle

This notebook is fail-closed. It validates tests, governance evidence, hashes, leakage, and a five-step CUDA smoke run before full training. It never prints `HF_TOKEN`, and upload remains disabled until a human changes the explicit switch.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys

REPOSITORY = "https://github.com/oluwafemidiakhoa/gaialab-naija-assistant.git"
GIT_REF = os.environ.get("GAIALAB_GIT_REF", "agent/professional-training-pipeline")
WORK = Path("/kaggle/working")
REPO = WORK / "gaialab-naija-assistant"
CANDIDATE = Path(os.environ.get("GAIALAB_CANDIDATE_DIR", REPO / "data/release_candidates/v0.7-rc3"))
TRAIN_FILE = CANDIDATE / "training.jsonl"
VALIDATION_FILE = CANDIDATE / "validation.jsonl"
BENCHMARK_FILE = CANDIDATE / "held_out_benchmark.jsonl"
FULL_OUTPUT = WORK / "gaialab-v0.7.0-rc.3-lora"
RUN_FULL_TRAINING = False
PUSH_TO_HUB = False
HUB_MODEL_ID = "oluwafemidiakhoa/gaialab-naija-assistant-v0.7.0-rc.3-lora"

def command(*args, cwd=None):
    print("+", " ".join(str(arg) for arg in args))
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

if not REPO.exists():
    command("git", "clone", REPOSITORY, REPO)
else:
    command("git", "fetch", "--all", "--tags", cwd=REPO)
command("git", "checkout", GIT_REF, cwd=REPO)
command("git", "rev-parse", "HEAD", cwd=REPO)

In [ ]:
# Pinned, non-quantized training dependencies.
command(sys.executable, "-m", "pip", "install", "-r", "requirements-training.txt", cwd=REPO)

In [ ]:
# Read the optional token from Kaggle Secrets. Never display its value.
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    token = None
if token:
    os.environ["HF_TOKEN"] = token
print("HF_TOKEN available:", bool(token))

In [ ]:
# Hardware and full repository verification. Any failure stops the notebook.
import torch
print({"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.cuda.is_available(), "gpu_count": torch.cuda.device_count()})
if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU accelerator is required")
command(sys.executable, "-m", "pytest", "-q", cwd=REPO)
VALIDATED = WORK / "validated-v0.6"
command(sys.executable, "-m", "src.validate_dataset", "data/releases/v0.6/v0.6.jsonl", "--output-dir", VALIDATED, cwd=REPO)

In [ ]:
# Inspect governed candidate metadata and exact split counts before execution.
if not CANDIDATE.is_dir():
    raise RuntimeError("Attach the immutable v0.7-rc3 package as a private Kaggle Dataset and set GAIALAB_CANDIDATE_DIR")
manifest = json.loads((CANDIDATE / "release_candidate_manifest.json").read_text())
print(json.dumps(manifest, indent=2, sort_keys=True))
for name in ("training.jsonl", "validation.jsonl", "held_out_benchmark.jsonl"):
    path = CANDIDATE / name
    count = sum(bool(line.strip()) for line in path.read_text(encoding="utf-8").splitlines())
    print(name, count)
if manifest.get("eligible_count", 0) < 1:
    raise RuntimeError("Candidate has no eligible records; do not train or bypass governance")

In [ ]:
# Governance dry run: no model is loaded.
DRY_OUTPUT = WORK / "gaialab-v0.7.0-rc.3-dry-run"
command(sys.executable, "scripts/train_governed_lora.py", "--config", "configs/training/v0.7.0-rc.3.yaml", "--train-file", TRAIN_FILE, "--validation-file", VALIDATION_FILE, "--output-dir", DRY_OUTPUT, "--dry-run", cwd=REPO)
print((DRY_OUTPUT / "training_manifest.json").read_text())

In [ ]:
# Five-step real CUDA smoke training in a separate directory.
SMOKE_OUTPUT = WORK / "gaialab-v0.7.0-rc.3-smoke"
command(sys.executable, "scripts/train_governed_lora.py", "--config", "configs/training/v0.7.0-rc.3.yaml", "--train-file", TRAIN_FILE, "--validation-file", VALIDATION_FILE, "--output-dir", SMOKE_OUTPUT, "--smoke-test", cwd=REPO)
smoke_manifest = json.loads((SMOKE_OUTPUT / "training_manifest.json").read_text())
if smoke_manifest["training_completion_status"] != "smoke_test_completed":
    raise RuntimeError("Smoke training did not complete; full training is blocked")

In [ ]:
# Full training requires a separate explicit human switch.
if not RUN_FULL_TRAINING:
    raise RuntimeError("Set RUN_FULL_TRAINING=True only after inspecting the smoke manifest")
train_args = [sys.executable, "scripts/train_governed_lora.py", "--config", "configs/training/v0.7.0-rc.3.yaml", "--train-file", TRAIN_FILE, "--validation-file", VALIDATION_FILE, "--output-dir", FULL_OUTPUT]
command(*train_args, cwd=REPO)

### Resume after interruption

Set `CHECKPOINT` below to an existing `checkpoint-*` directory. Do not combine resume with overwrite.

In [ ]:
CHECKPOINT = None  # e.g. FULL_OUTPUT / "checkpoint-25"
if CHECKPOINT is not None:
    command(sys.executable, "scripts/train_governed_lora.py", "--config", "configs/training/v0.7.0-rc.3.yaml", "--train-file", TRAIN_FILE, "--validation-file", VALIDATION_FILE, "--output-dir", FULL_OUTPUT, "--resume-from-checkpoint", CHECKPOINT, cwd=REPO)

In [ ]:
# Held-out evaluation. Training-file input is mandatory for leakage detection.
EVAL_OUTPUT = WORK / "gaialab-v0.7.0-rc.3-evaluation"
command(sys.executable, "scripts/evaluate_governed_adapter.py", "--release-version", "v0.7.0-rc.3", "--adapter-dir", FULL_OUTPUT / "adapter", "--training-file", TRAIN_FILE, "--evaluation-file", BENCHMARK_FILE, "--output-dir", EVAL_OUTPUT, cwd=REPO)
print((EVAL_OUTPUT / "evaluation_summary.json").read_text())

In [ ]:
# Plot recorded Trainer losses without manufacturing missing metrics.
import matplotlib.pyplot as plt
states = sorted(FULL_OUTPUT.glob("checkpoint-*/trainer_state.json"))
if not states:
    print("No trainer_state.json found")
else:
    history = json.loads(states[-1].read_text()).get("log_history", [])
    train = [(item["step"], item["loss"]) for item in history if "loss" in item]
    valid = [(item["step"], item["eval_loss"]) for item in history if "eval_loss" in item]
    if train: plt.plot(*zip(*train), label="training loss")
    if valid: plt.plot(*zip(*valid), label="validation loss")
    plt.xlabel("step"); plt.ylabel("loss"); plt.legend(); plt.show()

In [ ]:
# Optional upload is separate, explicit, and requires HF_TOKEN.
if PUSH_TO_HUB:
    if not os.environ.get("HF_TOKEN"):
        raise RuntimeError("HF_TOKEN Kaggle Secret is required for upload")
    from huggingface_hub import HfApi
    HfApi().upload_folder(repo_id=HUB_MODEL_ID, folder_path=FULL_OUTPUT / "adapter", repo_type="model")
else:
    print("Upload disabled")

In [ ]:
# Create a downloadable archive inside Kaggle working storage.
import shutil
archive = shutil.make_archive(str(WORK / "gaialab-v0.7.0-rc.3-artefacts"), "zip", root_dir=WORK, base_dir=FULL_OUTPUT.name)
print("Archive:", archive)